# Week 7: Iris Species Model Deployment
**AnalystLab Africa Internship**


## Project Overview

In this notebook, I will take the Iris species classification model I built and trained in previous weeks and deploy it as a web API. This means anyone- a developer, a mobile app, or another system will be able to send flower measurements to my API and receive a species prediction in return, without needing to run any Python code themselves.

My deployment workflow has three major stages:
1. **Save the trained model** using Joblib so it can be reloaded later without retraining
2. **Build a REST API** using Flask that accepts input data and returns predictions
3. **Test the API** to confirm it returns accurate results

I am choosing to use the **Logistic Regression** model because it outperformed the Random Forest Classifier in my Irir model evaluation, it achieved higher accuracy and made fewer misclassifications on the test set.

## Import Libraries

In [4]:
# I start by importing every library I will need for this project.
# These cover data loading, model training, model saving, and API building.

import numpy as np
import pandas as pd
import joblib
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from flask import Flask, request, jsonify

print("All libraries imported successfully.")

All libraries imported successfully.


## Load Dataset

In [5]:
# I load the Iris dataset. This is the same dataset I used to train the model previously.
# I need to retrain the model here so I can save it properly.

df = pd.read_csv('Iris.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nSpecies classes: {df['Species'].unique()}")
print(f"\nFirst 5 rows:")
df.head()

Dataset shape: (150, 6)

Column names: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']

Species classes: <StringArray>
['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Length: 3, dtype: str

First 5 rows:


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## Feature Selection

In [6]:
# I separate my features (X) from my target label (y).
# I drop the 'Id' column because it is just a row number — it carries no predictive information.
# The 4 flower measurements (SepalLength, SepalWidth, PetalLength, PetalWidth) are my input features.
# The 'Species' column is what I want the model to predict.

X = df.drop(['Id', 'Species'], axis=1)
y = df['Species']

print("Feature columns:", list(X.columns))
print("Target column: Species")
print(f"\nTotal samples: {len(df)}")

Feature columns: ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
Target column: Species

Total samples: 150


## Train-Test Split

In [7]:
# I split the data into training and testing sets using a 70/30 ratio.
# stratify=y ensures each species class is proportionally represented in both sets.
# random_state=42 ensures my results are reproducible, I get the same split every time I run this.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f"Training set size:  {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.0f}%)")
print(f"Testing set size:   {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.0f}%)")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())

Training set size:  105 samples (70%)
Testing set size:   45 samples (30%)

Class distribution in training set:
Species
Iris-versicolor    35
Iris-setosa        35
Iris-virginica     35
Name: count, dtype: int64


## Model Retraining

In [8]:
# I retrain the Logistic Regression model on the training data.
# I set max_iter=200 because the default (100 iterations) sometimes causes a convergence warning
# on the Iris dataset, increasing the limit gives the solver more time to find
# the optimal solution. I use random_state=42 for reproducibility.

lr_model = LogisticRegression(max_iter=200, random_state=42)
lr_model.fit(X_train, y_train)

print("Model training complete")
print(f"Model type: {type(lr_model).__name__}")

Model training complete
Model type: LogisticRegression


## Model Evaluation

In [9]:
# I verify the model's accuracy before saving it.
# It would be pointless to deploy a model that performs poorly.
# I want to confirm it still achieves the strong results I recorded previously.

y_pred = lr_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test Set Accuracy: {accuracy * 100:.2f}%")
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

Test Set Accuracy: 93.33%

Detailed Classification Report:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        15
Iris-versicolor       0.88      0.93      0.90        15
 Iris-virginica       0.93      0.87      0.90        15

       accuracy                           0.93        45
      macro avg       0.93      0.93      0.93        45
   weighted avg       0.93      0.93      0.93        45



### Model Accuracy Interpretation

My Logistic Regression model achieved strong results on the test set:

- **Overall Accuracy**: The model correctly classified the vast majority of the 45 test samples (30% of 150 total). An accuracy of **93.33%** means the model is making around **3 errors** out of every 45 predictions.

- **Precision**: For each species, precision tells me of all the times I predicted that species, how often was I right? High precision means few false positives.

- **Recall**: For each species, recall tells me of all the actual instances of that species, how many did I catch correctly? High recall means few false negatives.

- **Iris-setosa** is the easiest to classify: its petal measurements are so distinct from the other two species that the model always identifies it perfectly (precision and recall of 1.00).

- **Iris-versicolor and Iris-virginica** are more similar to each other, so the model occasionally confuses them. This matches what I observed in my previous pairplot, those two species overlap slightly in sepal measurements.

This performance level is solid and acceptable for deployment.

## Save the Trained Model with Joblib

In [10]:
# I save the trained model to disk using Joblib.
# Joblib is the recommended way to save scikit-learn models, it serializes the entire
# model object (coefficients, class labels, hyperparameters) into a single .pkl file.
# Once saved, I can load it at any time in the future without rerunning any training code.
# This is what makes deployment possible: the API will load this file and use it to make predictions.

model_path = 'iris_model.pkl'
joblib.dump(lr_model, model_path)

# Confirm the file was saved correctly
file_size = os.path.getsize(model_path)
print(f"Model saved successfully!")
print(f"File: {model_path}")
print(f"File size: {file_size} bytes ({file_size/1024:.1f} KB)")

Model saved successfully!
File: iris_model.pkl
File size: 1423 bytes (1.4 KB)


## Verifying the saved model

In [11]:
# I verify that the saved model can be reloaded and still makes correct predictions.
# This is a critical check, I want to make sure the file was not corrupted
# during saving. If the loaded model produces the same accuracy as the original,
# I know the save was successful.

loaded_model = joblib.load(model_path)
loaded_pred = loaded_model.predict(X_test)
loaded_accuracy = accuracy_score(y_test, loaded_pred)

print("Model reloaded from disk successfully!")
print(f"Reloaded Model Accuracy: {loaded_accuracy * 100:.2f}%")
print()

# Quick sanity test with one example prediction
sample = [[5.1, 3.5, 1.4, 0.2]]  # A known Iris-setosa sample
prediction = loaded_model.predict(sample)
print(f"Quick test prediction for {sample[0]}:")
print(f"Predicted species: {prediction[0]}")
print(f"(Expected: Iris-setosa — the model is working correctly.)")

Model reloaded from disk successfully!
Reloaded Model Accuracy: 93.33%

Quick test prediction for [5.1, 3.5, 1.4, 0.2]:
Predicted species: Iris-setosa
(Expected: Iris-setosa — the model is working correctly.)


C:\Users\Admin\PycharmProjects\Iris_Classifier_Deployment\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


## Build the Flask API

In [12]:
# I now write the Flask API code to a Python file called app.py.
# I write it as a file rather than running it here because Flask is a web server,
# it needs to run as a separate, always-on process, not inside a notebook cell.
#
# The API will have two endpoints:
#   GET  /health  — confirms the API is running
#   POST /predict — accepts flower measurements and returns a species prediction

api_code = '''
# app.py — Iris Species Prediction API
# I already import Flask to build the web server, joblib to load my saved model,
# and numpy to format the input before passing it to the model.


# I create the Flask application object.
# __name__ tells Flask where to find resources relative to this file.
app = Flask(__name__)

# I load the saved Logistic Regression model into memory when the server starts.
# This means the model is ready to make predictions the moment the first request arrives —
# I don't have to reload it for every single request, which keeps the API fast.
model = joblib.load('iris_model.pkl')

print("Iris prediction model loaded successfully.")


# --- Endpoint 1: Health Check ---
# This endpoint tells anyone who checks whether the API server is alive.
# It is useful for monitoring tools and for verifying the server started correctly.
@app.route('/health', methods=['GET'])
def health_check():
    return jsonify({
        "status": "running",
        "model": "Logistic Regression — Iris Species Classifier",
        "message": "API is healthy and ready to receive predictions."
    })


# --- Endpoint 2: Prediction ---
# This is the main endpoint. A client sends four flower measurements as JSON,
# and I return the predicted species name along with confidence probabilities.
@app.route('/predict', methods=['POST'])
def predict():
    # I retrieve the JSON body sent by the client.
    data = request.get_json()

    # I validate the incoming data to make sure all 4 required features are present.
    # If any feature is missing, I return a 400 error with a helpful message
    # so the user knows exactly what they need to fix.
    required_features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

    for feature in required_features:
        if feature not in data:
            return jsonify({
                "error": f"Missing required field: {feature}",
                "required_fields": required_features
            }), 400

    # I extract the values and reshape them into a 2D array.
    # scikit-learn's predict() expects a 2D array (even for a single sample),
    # so I use [[ ]] to wrap my 4 values as a single-row matrix.
    features = np.array([[
        data['SepalLengthCm'],
        data['SepalWidthCm'],
        data['PetalLengthCm'],
        data['PetalWidthCm']
    ]])

    # I use the loaded model to make the prediction.
    prediction = model.predict(features)[0]

    # I also get the probability for each class so I can show confidence scores.
    # predict_proba returns the probability that the input belongs to each species.
    probabilities = model.predict_proba(features)[0]
    class_names = model.classes_

    # I build a confidence dictionary that maps each species to its probability.
    confidence_scores = {
        cls: round(float(prob) * 100, 2)
        for cls, prob in zip(class_names, probabilities)
    }

    # I return the prediction as a clean JSON response.
    return jsonify({
        "predicted_species": prediction,
        "confidence_scores_percent": confidence_scores,
        "input_received": data
    })


# I run the Flask development server on port 5000.
# debug=True makes it easier to spot errors during testing — it shows detailed error messages
# and automatically restarts the server when I edit the code.
# NOTE: For production deployment, debug should be set to False.
if __name__ == '__main__':
    app.run(debug=True, port=5000)
'''

# I write the code to app.py
with open('app.py', 'w') as f:
    f.write(api_code.strip())

print("app.py created successfully!")
print("\nFile contents preview:")
print(api_code[:500] + '...')

app.py created successfully!

File contents preview:

# app.py — Iris Species Prediction API
# I already import Flask to build the web server, joblib to load my saved model,
# and numpy to format the input before passing it to the model.


# I create the Flask application object.
# __name__ tells Flask where to find resources relative to this file.
app = Flask(__name__)

# I load the saved Logistic Regression model into memory when the server starts.
# This means the model is ready to make predictions the moment the first request arrives —
# I don...


### API Design Explanation

I structured the Flask API with two endpoints intentionally:

**`GET /health`** — A health check endpoint is standard practice in any API. It lets me (or any monitoring service) quickly confirm the server is up and the model loaded correctly, without making an actual prediction. This is important in production environments where I would want to be alerted if the server goes down.

**`POST /predict`** — I use POST (not GET) for predictions because I am sending data *to* the server (the flower measurements), not just requesting information. This aligns with REST API best practices. The endpoint:
1. Validates that all 4 required features are present
2. Reshapes the data into the format scikit-learn expects
3. Returns both the predicted species and confidence probabilities — so users can see *how confident* the model is, not just what it predicted

I also included **input validation** (checking for missing fields) with a clear error message. This makes the API robust: if someone forgets a field, they get a helpful error instead of a cryptic Python crash.

## Test the API

To test the API, I need to:
1. Start the Flask server by running `python app.py` in a terminal
2. Send test requests using the `requests` library or curl

The cells below show how to test the API once the server is running. I also include a standalone test script that I can run from the terminal.

In [13]:
# I write a test script that will send requests to my Flask API.
# This script tests both the health endpoint and the prediction endpoint
# with several sample inputs, including one invalid request (missing a field)
# to confirm my error handling works correctly.

test_code = '''
# test_api.py — Test script for the Iris Prediction API
# Run this after starting app.py in a separate terminal.

import requests
import json

BASE_URL = "http://127.0.0.1:5000"


def print_response(label, response):
    """Helper function to display API responses clearly."""
    print(f"\n{'='*50}")
    print(f"TEST: {label}")
    print(f"Status Code: {response.status_code}")
    print(f"Response:")
    print(json.dumps(response.json(), indent=2))


# --- Test 1: Health Check ---
# I verify the server is running and the model loaded correctly.
response = requests.get(f"{BASE_URL}/health")
print_response("Health Check", response)


# --- Test 2: Iris-setosa prediction ---
# These measurements (small sepal, very small petal) are characteristic of Iris-setosa.
# I expect the model to predict Iris-setosa with very high confidence.
setosa_input = {
    "SepalLengthCm": 5.1,
    "SepalWidthCm": 3.5,
    "PetalLengthCm": 1.4,
    "PetalWidthCm": 0.2
}
response = requests.post(f"{BASE_URL}/predict", json=setosa_input)
print_response("Predict Iris-setosa", response)


# --- Test 3: Iris-versicolor prediction ---
# These medium-sized measurements are typical of Iris-versicolor.
versicolor_input = {
    "SepalLengthCm": 6.0,
    "SepalWidthCm": 2.9,
    "PetalLengthCm": 4.5,
    "PetalWidthCm": 1.5
}
response = requests.post(f"{BASE_URL}/predict", json=versicolor_input)
print_response("Predict Iris-versicolor", response)


# --- Test 4: Iris-virginica prediction ---
# These large measurements are typical of Iris-virginica.
virginica_input = {
    "SepalLengthCm": 6.9,
    "SepalWidthCm": 3.1,
    "PetalLengthCm": 5.4,
    "PetalWidthCm": 2.1
}
response = requests.post(f"{BASE_URL}/predict", json=virginica_input)
print_response("Predict Iris-virginica", response)


# --- Test 5: Missing field (error handling test) ---
# I intentionally leave out PetalWidthCm to test that my API returns
# a proper 400 error with a helpful message instead of crashing.
incomplete_input = {
    "SepalLengthCm": 5.1,
    "SepalWidthCm": 3.5,
    "PetalLengthCm": 1.4
    # PetalWidthCm intentionally omitted
}
response = requests.post(f"{BASE_URL}/predict", json=incomplete_input)
print_response("Missing Field Error (Expected 400)", response)


print("\n" + "="*50)
print("All tests complete!")
'''

with open('test_api.py', 'w') as f:
    f.write(test_code.strip())

print("test_api.py created successfully")

test_api.py created successfully


In [14]:
# I demonstrate what the API responses look like using direct model calls.
# This simulates the exact output the API would return, so I can verify correctness
# without needing to run a separate server process inside this notebook.

import json

# The 3 test samples — one per species
test_cases = [
    {"label": "Iris-setosa",     "features": [5.1, 3.5, 1.4, 0.2]},
    {"label": "Iris-versicolor", "features": [6.0, 2.9, 4.5, 1.5]},
    {"label": "Iris-virginica",  "features": [6.9, 3.1, 5.4, 2.1]},
]

feature_names = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

print("SIMULATED API RESPONSES")
print("=" * 55)

for case in test_cases:
    features_array = np.array([case['features']])
    prediction = loaded_model.predict(features_array)[0]
    probabilities = loaded_model.predict_proba(features_array)[0]
    class_names = loaded_model.classes_

    confidence_scores = {
        cls: round(float(prob) * 100, 2)
        for cls, prob in zip(class_names, probabilities)
    }

    input_data = dict(zip(feature_names, case['features']))

    response = {
        "predicted_species": prediction,
        "confidence_scores_percent": confidence_scores,
        "input_received": input_data
    }

    print(f"\nExpected species: {case['label']}")
    print(json.dumps(response, indent=2))
    print("-" * 55)

SIMULATED API RESPONSES

Expected species: Iris-setosa
{
  "predicted_species": "Iris-setosa",
  "confidence_scores_percent": {
    "Iris-setosa": 97.7,
    "Iris-versicolor": 2.3,
    "Iris-virginica": 0.0
  },
  "input_received": {
    "SepalLengthCm": 5.1,
    "SepalWidthCm": 3.5,
    "PetalLengthCm": 1.4,
    "PetalWidthCm": 0.2
  }
}
-------------------------------------------------------

Expected species: Iris-versicolor
{
  "predicted_species": "Iris-versicolor",
  "confidence_scores_percent": {
    "Iris-setosa": 0.79,
    "Iris-versicolor": 79.2,
    "Iris-virginica": 20.01
  },
  "input_received": {
    "SepalLengthCm": 6.0,
    "SepalWidthCm": 2.9,
    "PetalLengthCm": 4.5,
    "PetalWidthCm": 1.5
  }
}
-------------------------------------------------------

Expected species: Iris-virginica
{
  "predicted_species": "Iris-virginica",
  "confidence_scores_percent": {
    "Iris-setosa": 0.01,
    "Iris-versicolor": 10.6,
    "Iris-virginica": 89.39
  },
  "input_received": {


C:\Users\Admin\PycharmProjects\Iris_Classifier_Deployment\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\Admin\PycharmProjects\Iris_Classifier_Deployment\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\Admin\PycharmProjects\Iris_Classifier_Deployment\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\Admin\PycharmProjects\Iris_Classifier_Deployment\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\Admin\PycharmProjects\Iris_Classifier_Deploymen

### Test Results Interpretation

The simulated API responses show exactly what a client application would receive:

**Iris-setosa** (input: sepal 5.1×3.5 cm, petal 1.4×0.2 cm): The model predicts this species with near 100% confidence. This makes sense — Iris-setosa has distinctly small petals compared to the other two species. The pairplot from my EDA showed a completely separate cluster for setosa, so the model has no trouble identifying it.

**Iris-versicolor** (input: sepal 6.0×2.9 cm, petal 4.5×1.5 cm): The model predicts versicolor, though with somewhat lower confidence than setosa. This reflects the slight overlap between versicolor and virginica that I observed in Week 6 — the model is appropriately less certain when measurements fall in the mid-range.

**Iris-virginica** (input: sepal 6.9×3.1 cm, petal 5.4×2.1 cm): The model correctly predicts virginica. Larger petal measurements — especially petal length above 5.0 cm — are strong indicators of this species.

The **confidence scores** are especially useful: they show not just *what* the model predicts, but *how sure* it is. A prediction with 99% confidence is much more trustworthy than one with 55% confidence, and returning this information gives users the context they need to interpret results appropriately.